In [0]:

# Notebook : helper_functions
# Purpose  : Reusable helper functions for Procurement Project
# Author   : V R  Mutyala


from pyspark.sql import DataFrame
from pyspark.sql.functions import *
from pyspark.sql.window import Window


# Preview Data
def preview(df: DataFrame, table_name: str, rows: int = 10):
    """
    Display row count and sample records.
    """

    print("=" * 60)
    print(f"Table : {table_name}")
    print(f"Rows  : {df.count()}")
    print("=" * 60)

    display(df.limit(rows))

#Record Count
def record_count(df: DataFrame, table_name: str):
    """
    Print record count.
    """

    print(f"{table_name} : {df.count():,} records")

# Null Check 

def null_blank_check(df: DataFrame):

    columns = df.columns

    return df.select([
        count(
            when(
                col(c).isNull() |
                (trim(col(c).cast("string")) == ""),
                c
            )
        ).alias(c)
        for c in columns
    ])

# Duplicate Check 
def duplicate_check(df: DataFrame, key_column: str):

    window_spec = Window.partitionBy(key_column).orderBy(key_column)

    duplicate_df = (
        df.withColumn(
            "row_num",
            row_number().over(window_spec)
        )
        .filter(col("row_num") > 1)
        .drop("row_num")
    )

    return duplicate_df

#Sepreate Duplicates
def split_duplicates(df: DataFrame, key_columns: list):

    duplicate_keys = (
        df.groupBy(key_columns)
          .count()
          .filter(col("count") > 1)
          .drop("count")
    )

    duplicates = df.join(
        duplicate_keys,
        key_columns,
        "inner"
    )

    valid_records = df.join(
        duplicate_keys,
        key_columns,
        "left_anti"
    )

    return valid_records, duplicates

#Write Delta table
def write_delta(
    df: DataFrame,
    table_name: str,
    mode: str = "overwrite",
    overwrite_schema: bool = True
):
    """
    Write DataFrame to a Delta table.

    Parameters
    ----------
    df : Spark DataFrame
    table_name : Target Delta table
    mode : overwrite | append
    overwrite_schema : Update schema when it changes
    """

    (
        df.write
          .format("delta")
          .mode(mode)
          .option("overwriteSchema", str(overwrite_schema).lower())
          .saveAsTable(table_name)
    )

    print(f"✓ Table written successfully : {table_name}")

# Read Delta table
def read_delta(table_name: str) -> DataFrame:
    """
    Read a Delta table.
    """

    return spark.table(table_name)

# Add Audit columns 
def add_audit_columns(
    df: DataFrame,
    source_file: str,
    pipeline_layer: str
):

    return (
        df
        .withColumn("load_timestamp", current_timestamp())
        .withColumn("source_file", lit(source_file))
        .withColumn("pipeline_layer", lit(pipeline_layer))
        .withColumn("created_by", lit("Databricks"))
    )


# Validate Delta Table

def validate_table(table_name: str):
    """
    Display record count and sample data from a Delta table.
    """

    df = spark.table(table_name)

    print("=" * 60)
    print(f"Table : {table_name}")
    print(f"Rows  : {df.count():,}")
    print("=" * 60)

    display(df)
    
